### Preprocessing of clinical data

`csv`: clinical data

1. patient ID (PatientID)

2. age (patient's age)

3. Size and extent of the primary tumor - Tumor (T) (Clinical.T.Stage - T1, T2, T3, T4 → the higher the number, the larger or more invasive the tumor) 

4. Whether the cancer has spread to regional lymph nodes - Lymph nodes (N) (Clinical.N.Stage - N0 → no lymph nodes affected. N1, N2, N3 → increasing number of affected lymph nodes and/or greater extent. Nx → lymph nodes could not be assessed)

5. Whether the cancer has spread to other organs - Metastasis (M) (Clinical.M.Stage - M0 → no distant metastasis. M1 → presence of metastasis. Mx → could not be determined)

6. Overall disease stage (T + N + M) - (Overall.Stage - T1 + N0 + M0 = Stage I, T2 + N1 + M0 = Stage II, T3 + N2 + M0 = Stage III, Any T + Any N + M1 = Stage IV

7. Tumor histological type (Histology - LUAD, LUSC)

8. Gender (male and female)

9. Survival time, i.e., the number of months the patient survived until the outcome (Survival.time)

10. Indicates whether the event occurred, which may be death, recurrence, or censored (patient alive, lost to follow-up, etc.) (deadstatus.event)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
df_clinical = pd.read_csv("/content/drive/MyDrive/NSCLC/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv")

In [ ]:
# I had to delete patient 128 because I was unable to obtain the 3D model for this patient

df_clinical = df_clinical[df_clinical["PatientID"] != "LUNG1-128"]

In [ ]:
df_clinical.isnull().sum()

In [ ]:
df_clinical.groupby("deadstatus.event").size()
# 0 = 48, 1 = 373

#### Handling Missing Values

In [ ]:
df_clinical["age"] = df_clinical["age"].fillna(df_clinical["age"].median())


In [ ]:
df_clinical["clinical.T.Stage"] = df_clinical["clinical.T.Stage"].fillna(df_clinical["clinical.T.Stage"].mode()[0])
df_clinical["Overall.Stage"] = df_clinical["Overall.Stage"].fillna(df_clinical["Overall.Stage"].mode()[0])
df_clinical["Histology"] = df_clinical["Histology"].fillna(df_clinical["Histology"].mode()[0])


#### Coding of Categorical Attributes

In [ ]:
df_clinical["Overall.Stage"] = (
    df_clinical["Overall.Stage"]
    .astype(str)
    .str.strip()
    .str.upper()
)


In [ ]:
# Label encoding

stage_map = {
    "I": 1,
    "II": 2,
    "IIIA": 3,
    "IIIB": 4
}

df_clinical["Overall.Stage"] = df_clinical["Overall.Stage"].map(stage_map)


# One-hot encoding

df_clinical = pd.get_dummies(df_clinical, columns=["gender", "Histology"], drop_first=True)


In [ ]:
bool_cols = df_clinical.select_dtypes(include='bool').columns
df_clinical[bool_cols] = df_clinical[bool_cols].astype(int)


#### Normalization

The following attributes had to be normalized: `age`, `clinical.T.Stage`, `Clinical.N.Stage`, `Clinical.M.Stage`

In [ ]:
norm = [
    "age",
    "clinical.T.Stage",
    "Clinical.N.Stage",
    "Clinical.M.Stage"

]

scaler = MinMaxScaler()

df_clinical[norm] = scaler.fit_transform(df_clinical[norm])

df_clinical[norm].head(100)


In [ ]:
df_clinical.to_csv("/content/drive/MyDrive/NSCLC/df_clinical.csv", index=False)